In [ ]:
!nvidia-smi

In [ ]:
# Cài đặt thư viện (tắt output cho gọn bằng -q)
!pip install ultralytics -q

# Kiểm tra cài đặt và phần cứng
import ultralytics
ultralytics.checks()

In [ ]:
!ls -R /kaggle/input/licenseplates | head -n 20

In [ ]:
import yaml
import os

# 1. Định nghĩa đường dẫn tuyệt đối tới thư mục ảnh
train_path = '/kaggle/input/licenseplates/images/train'
val_path = '/kaggle/input/licenseplates/images/val'

# 2. Kiểm tra xem thư mục labels có tồn tại tương ứng không
label_check = '/kaggle/input/licenseplates/labels/train'
if os.path.exists(label_check):
    print(f" Đã tìm thấy thư mục nhãn tại: {label_check}")
else:
    print(f" CẢNH BÁO: Không tìm thấy thư mục nhãn tại {label_check}. Vui lòng kiểm tra lại cấu trúc thư mục labels.")

# 3. Nội dung file cấu hình
data_config = {
    'train': train_path,
    'val': val_path,
    'nc': 2,            # Số lượng lớp (classes)
    'names': ['LpD', 'LpV'] 
}

# 4. Ghi file data.yaml vào thư mục làm việc hiện tại
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("✅ Đã tạo file data.yaml thành công!")

In [ ]:
from ultralytics import YOLO

# Load model YOLO11 nano segmentation (bản nhẹ nhất, phù hợp test)
model = YOLO('yolo11n-seg.pt')

# Bắt đầu train
results = model.train(
    data='/kaggle/working/data.yaml', 
    epochs=50,                        
    imgsz=640,                        # Kích thước ảnh chuẩn
    batch=16,                         
    name='yolo11_license_plate',      # Tên thư mục lưu kết quả
    device=0,                         
    close_mosaic=10                   
)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob
import random

# Đường dẫn tới model tốt nhất vừa train xong
best_model_path = '/kaggle/working/runs/segment/yolo11_license_plate/weights/best.pt'
model = YOLO(best_model_path)

# Lấy ngẫu nhiên 1 ảnh trong tập validation để test
val_images = glob.glob('/kaggle/input/licenseplates/images/val/*.png')
random_img = random.choice(val_images)

# Dự đoán
print(f"Đang test trên ảnh: {random_img}")
results = model.predict(random_img, conf=0.4, save=True)

# Hiển thị ảnh kết quả (YOLO lưu ảnh kết quả vào thư mục runs/segment/predict...)
import os
latest_predict_folder = max(glob.glob('/kaggle/working/runs/segment/predict*'), key=os.path.getmtime)
predicted_img_path = os.path.join(latest_predict_folder, os.path.basename(random_img))

if os.path.exists(predicted_img_path):
    img = Image.open(predicted_img_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Không tìm thấy ảnh kết quả. Kiểm tra lại folder runs/segment/predict")

In [ ]:
from ultralytics import YOLO
import torch

# 1. Load model tốt nhất đã train
# Lưu ý: Đảm bảo đường dẫn này đúng với nơi lưu file best.pt của bạn
model_path = '/kaggle/working/runs/segment/yolo11_license_plate/weights/best.pt'
model = YOLO(model_path)

print("="*20 + " BÁO CÁO HIỆU NĂNG " + "="*20)

# --- PHẦN 1: Params (M) và FLOPs (G) ---
# Hàm info() sẽ in ra số layers, params và FLOPs
print("\n[1] Model Complexity:")
model.info(detailed=True)

# --- PHẦN 2: mAP50 và mAP50-95 ---
# Chạy đánh giá trên tập validation
print("\n[2] Accuracy Metrics (trên tập Val):")
metrics = model.val(data='/kaggle/working/data.yaml', split='val', device=0, verbose=False)

# Vì là bài toán Segmentation, ta có cả chỉ số Box (khung) và Mask (đa giác)
# Thông thường với bài toán biển số, ta quan tâm Mask hơn.
print(f"\n--- Mask (Segmentation) Metrics ---")
print(f"👉 mAP50    : {metrics.seg.map50:.4f} (Độ chính xác tại IoU 0.5)")
print(f"👉 mAP50-95 : {metrics.seg.map:.4f}   (Độ chính xác trung bình toàn dải)")

print(f"\n--- Box (Detection) Metrics ---")
print(f"👉 mAP50    : {metrics.box.map50:.4f}")
print(f"👉 mAP50-95 : {metrics.box.map:.4f}")

# --- PHẦN 3: Latency (Độ trễ trên T4 GPU) ---
# metrics.speed trả về dictionary chứa thời gian trung bình (ms) cho mỗi ảnh
# Bao gồm: preprocess (tiền xử lý), inference (chạy mô hình), postprocess (hậu xử lý/NMS)
speed = metrics.speed
t_inference = speed['inference']
t_total = speed['inference'] + speed['postprocess'] # Tổng thời gian xử lý chính

print("\n[3] Latency (Tốc độ trên Tesla T4 GPU):")
print(f"👉 Inference Time : {t_inference:.2f} ms/img")
print(f"👉 Total Latency  : {t_total:.2f} ms/img (Inference + NMS)")
print(f"👉 FPS Ước tính   : {1000/t_total:.1f} FPS")

print("="*60)